# 1. Data Exploration & Visualization (MetaMathQA)

**Objective:** Load the training dataset (**MetaMathQA**), parse it into our `(Prompt, CoT, Solution)` format, and visualize its properties.



## 1.1 Environment Setup and Repository Cloning

To ensure reproducibility, this section automates the setup of the working environment:
1. **Google Drive Integration:** Mounts your personal Drive to store persistent data (checkpoints and processed datasets).
2. **Project Structure:** Automatically creates a `DLAI` folder in your Drive.
3. **Dependency Management:** Installs the `uv` package manager and resolves all requirements defined in `pyproject.toml`.
4. **Source Code:** Clones the `llama` branch from our GitHub repository to provide access to the `src` module and configuration files.

**Note for Evaluators:** Please authorize the Google Drive mount when prompted to allow the notebook to save and retrieve project files.

In [ ]:
import os, sys

# 1. Mount Google Drive
# Evaluators will need to accept the pop-up to connect their Drive
from google.colab import drive
drive.mount('/content/drive')

# 2. Setup directories on Drive
# Create the DLAI folder if it doesn't exist on their Drive
DRIVE_PROJECT_PATH = "/content/drive/MyDrive/DLAI"
if not os.path.exists(DRIVE_PROJECT_PATH):
    os.makedirs(DRIVE_PROJECT_PATH, exist_ok=True)
    print(f"Created project folder at: {DRIVE_PROJECT_PATH}")

# 3. UV Installation
# We use UV for much faster dependency management than standard pip
!curl -LsSf https://astral.sh/uv/install.sh | sh
os.environ['PATH'] = f"{os.path.expanduser('~')}/.cargo/bin:" + os.environ['PATH']

# 4. Clone the Repository (Branch: llama)
# If the local folder doesn't exist, clone the specific branch
%cd /content
if not os.path.exists("DLAI"):
    !git clone --branch llama https://github.com/irene-30/DLAI.git
else:
    print("Repo already exists, pulling latest changes...")
    !git -C DLAI pull

# 5. Synchronize pyproject.toml
# Copy the pyproject.toml from the cloned repo to the Drive folder (if necessary)
# or vice versa, to ensure that UV reads the correct dependencies.
!cp /content/DLAI/pyproject.toml {DRIVE_PROJECT_PATH}/pyproject.toml

# 6. Install dependencies via pyproject.toml
# This command reads the .toml file and installs everything necessary
%cd /content/DLAI
!uv pip install -e . --system

# 7. Add to the system path to allow imports from 'src'
sys.path.append("/content/DLAI")
%cd /content

print("✅ Setup completed successfully!")

Mounted at /content/drive
downloading uv 0.11.14 x86_64-unknown-linux-gnu
installing to /usr/local/bin
  uv
  uvx
everything's installed!
/content
Cloning into 'DLAI'...
remote: Enumerating objects: 640, done.
remote: Counting objects: 100% (194/194), done.
remote: Compressing objects: 100% (194/194), done.
remote: Total 640 (delta 136), reused 0 (delta 0), pack-reused 446 (from 2)
Receiving objects: 100% (640/640), 281.40 KiB | 9.08 MiB/s, done.
Resolving deltas: 100% (371/371), done.
/content/DLAI
Using Python 3.12.13 environment at: /usr
Resolved 88 packages in 1.23s
Prepared 6 packages in 2.42s
Uninstalled 3 packages in 83ms
Installed 6 packages in 31ms
 + bitsandbytes==0.49.2
 - datasets==4.0.0
 + datasets==4.8.5
 + dlai-metamath==0.1.0 (from file:///content/DLAI)
 - pyarrow==18.1.0
 + pyarrow==24.0.0
 - torchao==0.10.0
 + torchao==0.17.0
 + trl==1.4.0
/content
✅ Setup completed successfully!


## 1.2 Dataset Loading and Initial Exploration

We load the MetaMathQA dataset from Hugging Face and examine its internal structure. This dataset contains queries and detailed responses that will serve as the basis for our Chain-of-Thought (CoT) latent encoding.

In [ ]:
import torch
from datasets import load_dataset
from src.utils import get_llm_tokenizer, parse_sample

# Device Configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# 1. Load the MetaMathQA dataset
print("Loading MetaMathQA dataset...")
dataset = load_dataset("meta-math/MetaMathQA")['train']


Using device: cuda
Loading MetaMathQA dataset...


README.md: 0.00B [00:00, ?B/s]

MetaMathQA-395K.json:   0%|          | 0.00/396M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/395000 [00:00<?, ? examples/s]

## 1.3 Inspect a Single Sample

In [ ]:
sample = dataset[0]
print("--- RAW SAMPLE ---")
# MetaMath uses 'query' and 'response'
print(json.dumps(sample, indent=2) if 'json' in locals() else sample)

--- RAW SAMPLE ---
{'type': 'MATH_AnsAug', 'query': "Gracie and Joe are choosing numbers on the complex plane. Joe chooses the point $1+2i$. Gracie chooses $-1+i$. How far apart are Gracie and Joe's points?", 'original_question': "Gracie and Joe are choosing numbers on the complex plane. Joe chooses the point $1+2i$. Gracie chooses $-1+i$. How far apart are Gracie and Joe's points?", 'response': "The distance between two points $(x_1,y_1)$ and $(x_2,y_2)$ in the complex plane is given by the formula $\\sqrt{(x_2-x_1)^2+(y_2-y_1)^2}$.\nIn this case, Joe's point is $(1,2)$ and Gracie's point is $(-1,1)$.\nSo the distance between their points is $\\sqrt{((-1)-(1))^2+((1)-(2))^2}=\\sqrt{(-2)^2+(-1)^2}=\\sqrt{4+1}=\\sqrt{5}$.\nTherefore, Gracie and Joe's points are $\\boxed{\\sqrt{5}}$ units apart.\nThe answer is: \\sqrt{5}"}


## 1.4 Parse the Sample

Test our generic `parse_sample` utility.

In [ ]:
parsed = parse_sample(sample)

if parsed:
    prompt, cot, solution = parsed
    print("--- PARSED PROMPT (P) ---")
    print(f"{prompt}")

    print("\n--- PARSED CoT (C) ---")
    print(f"{cot}")

    print("\n--- PARSED SOLUTION (S) ---")
    print(f"{solution}")
else:
    print("Failed to parse sample.")
    print("Sample keys:", sample.keys())

--- PARSED PROMPT (P) ---
Question: Gracie and Joe are choosing numbers on the complex plane. Joe chooses the point $1+2i$. Gracie chooses $-1+i$. How far apart are Gracie and Joe's points?
Answer: 

--- PARSED CoT (C) ---
The distance between two points $(x_1,y_1)$ and $(x_2,y_2)$ in the complex plane is given by the formula $\sqrt{(x_2-x_1)^2+(y_2-y_1)^2}$.
In this case, Joe's point is $(1,2)$ and Gracie's point is $(-1,1)$.
So the distance between their points is $\sqrt{((-1)-(1))^2+((1)-(2))^2}=\sqrt{(-2)^2+(-1)^2}=\sqrt{4+1}=\sqrt{5}$.
Therefore, Gracie and Joe's points are $\boxed{\sqrt{5}}$ units apart.
The answer is: \sqrt{5}

--- PARSED SOLUTION (S) ---
 #### 5
